# mT5 Feature: Baseline 1

- Baseline 1: natural language -> label, direct inference, no fine-tuning.

## 1. Kaggle Path Configuration

In [1]:
from pathlib import Path

def find_data_dir():
    """Find the folder that directly contains folio_train.json, folio_valid.json, manual_test.json."""
    required = ["folio_train.json", "folio_valid.json", "manual_test.json"]

    candidates = [
        Path("/kaggle/input/mt5-feature/mt5_feature/data"),
        Path("/kaggle/input/mt5_feature/mt5_feature/data"),
        Path("/kaggle/input/mt5-feature/data"),
        Path("/kaggle/input/mt5_feature/data"),
        Path("mt5_feature/data"),
        Path("data"),
    ]

    for p in candidates:
        if p.exists() and all((p / name).exists() for name in required):
            return p

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for p in kaggle_input.glob("**/data"):
            if p.is_dir() and all((p / name).exists() for name in required):
                return p

    raise FileNotFoundError(
        "Could not find data directory. Please set DATA_DIR manually. "
        "Expected a folder containing folio_train.json, folio_valid.json, and manual_test.json."
    )

DATA_DIR = find_data_dir()
OUTPUT_DIR = Path("/kaggle/working/mt5_outputs") if Path("/kaggle/working").exists() else Path("mt5_feature/outputs")
CHECKPOINT_DIR = Path("/kaggle/working/mt5_checkpoints") if Path("/kaggle/working").exists() else Path("mt5_feature/checkpoints")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

FOLIO_TRAIN_PATH = DATA_DIR / "folio_train.json"
FOLIO_VALID_PATH = DATA_DIR / "folio_valid.json"
MANUAL_TEST_PATH = DATA_DIR / "manual_test.json"
HAO_TEST_PATH = DATA_DIR / "Hao.json"
FINAL_MANUAL_PATH = DATA_DIR / "final_manual_data.json"
DEFAULT_PROVER9_PATH = "/kaggle/working/Prover9-LADR-2026-4F/bin"

print("DATA_DIR =", DATA_DIR)
print("OUTPUT_DIR =", OUTPUT_DIR)
print("CHECKPOINT_DIR =", CHECKPOINT_DIR)
for name, path in [
    ("FOLIO_TRAIN_PATH", FOLIO_TRAIN_PATH),
    ("FOLIO_VALID_PATH", FOLIO_VALID_PATH),
    ("MANUAL_TEST_PATH", MANUAL_TEST_PATH),
    ("HAO_TEST_PATH", HAO_TEST_PATH),
    ("FINAL_MANUAL_PATH", FINAL_MANUAL_PATH),
]:
    print(f"{name} exists:", path.exists(), "|", path)

DATA_DIR = /kaggle/input/datasets/trnlnhtho/mt5-feature/mt5_feature/data
OUTPUT_DIR = /kaggle/working/mt5_outputs
CHECKPOINT_DIR = /kaggle/working/mt5_checkpoints
FOLIO_TRAIN_PATH exists: True | /kaggle/input/datasets/trnlnhtho/mt5-feature/mt5_feature/data/folio_train.json
FOLIO_VALID_PATH exists: True | /kaggle/input/datasets/trnlnhtho/mt5-feature/mt5_feature/data/folio_valid.json
MANUAL_TEST_PATH exists: True | /kaggle/input/datasets/trnlnhtho/mt5-feature/mt5_feature/data/manual_test.json
HAO_TEST_PATH exists: True | /kaggle/input/datasets/trnlnhtho/mt5-feature/mt5_feature/data/Hao.json
FINAL_MANUAL_PATH exists: True | /kaggle/input/datasets/trnlnhtho/mt5-feature/mt5_feature/data/final_manual_data.json


## 2. Install Dependencies

In [2]:
!pip install -q transformers datasets accelerate sentencepiece nltk regex pandas numpy

## 3. Imports

In [3]:
import copy
import gc
import inspect
import json
import os
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import regex as re
import torch
from datasets import Dataset
from nltk.inference import Prover9
from nltk.sem.logic import LogicParser
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 4. Data Audit

In [4]:
def load_json(path):
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError(f"{path} must contain a JSON list.")
    return data


def _as_text(value):
    if value is None:
        return ""
    if isinstance(value, list):
        return "\n".join(str(x).strip() for x in value if str(x).strip())
    return str(value).strip()


def normalize_example(example, index=0):
    premises = _as_text(example.get("nat_premises") or example.get("nat_premise") or example.get("premises"))
    conclusion = _as_text(example.get("nat_conclusion") or example.get("conclusion"))
    label = _as_text(example.get("label"))
    return {
        "id": _as_text(example.get("id") or example.get("story_id") or f"example_{index}"),
        "story_id": _as_text(example.get("story_id")),
        "premises": premises,
        "conclusion": conclusion,
        "label": label,
        "fol_premises": _as_text(example.get("fol_premises") or example.get("fol_premise")) or None,
        "fol_conclusion": _as_text(example.get("fol_conclusion")) or None,
    }


def audit_dataset(path, name):
    path = Path(path)
    print(f"\n## {name}: {path}")
    if not path.exists():
        print("MISSING")
        return None
    rows = load_json(path)
    normalized = [normalize_example(x, i) for i, x in enumerate(rows)]
    fields = list(rows[0].keys()) if rows else []
    labels = Counter(x["label"] for x in normalized)
    has_fol = any(x["fol_premises"] and x["fol_conclusion"] for x in normalized)
    print("num_examples:", len(rows))
    print("fields:", fields)
    print("label_distribution:", dict(labels))
    print("has_fol_fields:", has_fol)
    if normalized:
        preview = copy.deepcopy(normalized[0])
        for key in ["premises", "conclusion", "fol_premises", "fol_conclusion"]:
            if preview.get(key) and len(preview[key]) > 350:
                preview[key] = preview[key][:350] + "..."
        print("first_example_preview:")
        print(json.dumps(preview, ensure_ascii=False, indent=2))
    return normalized

folio_train_norm = audit_dataset(FOLIO_TRAIN_PATH, "FOLIO train")
folio_valid_norm = audit_dataset(FOLIO_VALID_PATH, "FOLIO valid")
manual_test_norm = audit_dataset(MANUAL_TEST_PATH, "Manual test")
if HAO_TEST_PATH.exists():
    hao_norm = audit_dataset(HAO_TEST_PATH, "Hao")
if FINAL_MANUAL_PATH.exists():
    final_manual_norm = audit_dataset(FINAL_MANUAL_PATH, "Final manual")


## FOLIO train: /kaggle/input/datasets/trnlnhtho/mt5-feature/mt5_feature/data/folio_train.json
num_examples: 856
fields: ['id', 'story_id', 'nat_premises', 'nat_conclusion', 'fol_premises', 'fol_conclusion', 'label']
label_distribution: {'True': 256, 'False': 206, 'Uncertain': 394}
has_fol_fields: True
first_example_preview:
{
  "id": "folio_train_0",
  "story_id": "folio_406",
  "premises": "All people who regularly drink coffee are dependent on caffeine.People regularly drink coffee, or they don't want to be addicted to caffeine, or both.No one who doesn't want to be addicted to caffeine is unaware that caffeine is a drug.Rina is either a student who is unaware that caffeine is a drug, or she is not a student and is she aware that caf...",
  "conclusion": "Rina doesn't want to be addicted to caffeine or is unaware that caffeine is a drug.",
  "label": "True",
  "fol_premises": "∀x (DrinkRegularly(x, coffee) → IsDependentOn(x, caffeine))\n∀x (DrinkRegularly(x, coffee) ∨ ¬WantToBeAddi

## 5. Engine, DataFilter, and filter2

In [5]:
class Engine:
    def __init__(self, prover9_path=DEFAULT_PROVER9_PATH):
        self.parser = LogicParser()
        self.prover9_path = prover9_path
        self.prover = Prover9()
        self.prover.config_prover9(prover9_path)

    def _translate_fol(self, fol_text: str):
        fol_text = re.sub(r"(?<=[a-zA-Z0-9])-(?=[a-zA-Z0-9])", "_", fol_text)
        replacements = {
            "∀": "all ",
            "∃": "exists ",
            "∧": "&",
            "∨": "|",
            "⊕": "^",
            "¬": "-",
            "→": "->",
            "⟷": "<->",
            "↔": "<->",
        }
        for k, v in replacements.items():
            fol_text = fol_text.replace(k, v)
        fol_text = re.sub(r"(all|exists)\s+([a-z0-9]+)\s*", r"\1 \2. ", fol_text)
        words = re.findall(r"\b[a-z][a-zA-Z0-9_]*\b", fol_text)
        reserved_words = {"all", "exists", "u", "v", "w", "x", "y", "z"}
        for w in set(words):
            if w not in reserved_words:
                fol_text = re.sub(fr"\b{w}\b", f"c_{w}", fol_text)
        return fol_text

    def _is_valid_fol(self, fol_list):
        try:
            for line in fol_list:
                if line.strip():
                    self.parser.parse(line)
            return True
        except Exception:
            return False

    def _check_conclusion(self, premises, conclusion):
        translated_premises = [self._translate_fol(p) for p in premises]
        translated_conclusion = self._translate_fol(conclusion)
        if not self._is_valid_fol(translated_premises) or not self._is_valid_fol([translated_conclusion]):
            raise ValueError("Invalid FOL syntax detected!")
        parsed_premises = [self.parser.parse(p) for p in translated_premises]
        parsed_conclusion = self.parser.parse(translated_conclusion)
        if self.prover.prove(parsed_conclusion, []):
            raise Exception("Error: The conclusion itself is True!")
        if self.prover.prove(parsed_conclusion.negate(), []):
            raise Exception("Error: The conclusion itself is False!")
        if self.prover.prove(None, parsed_premises):
            raise Exception("Error: The premises are inconsistent!")
        if self.prover.prove(parsed_conclusion, parsed_premises):
            return "True"
        if self.prover.prove(parsed_conclusion.negate(), parsed_premises):
            return "False"
        return "Uncertain"

    def check_conclusion(self, data):
        new_fol_list = []
        for fol in data.get("predictions", []):
            try:
                lines = [line.strip() for line in fol.get("fol", "").split("\n") if line.strip()]
                if len(lines) < 2:
                    continue
                fol = copy.deepcopy(fol)
                fol["label"] = self._check_conclusion(lines[:-1], lines[-1])
                new_fol_list.append(fol)
            except Exception:
                continue
        data = copy.deepcopy(data)
        data["predictions"] = new_fol_list
        return data


class DataFilter:
    def __init__(self):
        self.duplicate_count = 0
        self.syntax_error_count = 0
        self.length_ratio_count = 0

    def _is_valid_syntax(self, fol_str: str) -> bool:
        if not fol_str:
            return False
        stack = []
        matching_bracket = {")": "(", "]": "[", "}": "{"}
        for char in fol_str:
            if char in matching_bracket.values():
                stack.append(char)
            elif char in matching_bracket:
                if not stack or stack.pop() != matching_bracket[char]:
                    return False
        return len(stack) == 0

    def _is_valid_length_ratio(self, nat_str: str, fol_str: str, lowerbound_ratio: float = 0.5, upperbound_ratio: float = 2.0) -> bool:
        if not nat_str:
            return False
        ratio = len(fol_str) / len(nat_str)
        return lowerbound_ratio <= ratio <= upperbound_ratio

    def filter(self, entry: dict, lowerbound_ratio: float = 0.5, upperbound_ratio: float = 2.0) -> dict:
        seen_fols = set()
        valid_predictions = []
        nat_str = entry.get("natural", "")
        sentences = [s for s in nat_str.split(".") if s.strip()]
        nat_length = len(sentences)
        for pred in entry.get("predictions", []):
            fol = pred.get("fol", "")
            if fol in seen_fols:
                self.duplicate_count += 1
                continue
            if not self._is_valid_syntax(fol):
                self.syntax_error_count += 1
                continue
            unique_lines = []
            for line in [p.strip() for p in fol.split("\n") if p.strip()]:
                if line not in unique_lines:
                    unique_lines.append(line)
            if nat_length and len(unique_lines) != nat_length:
                continue
            if not self._is_valid_length_ratio(nat_str, fol, lowerbound_ratio, upperbound_ratio):
                self.length_ratio_count += 1
                continue
            seen_fols.add(fol)
            valid_predictions.append(pred)
        filtered = copy.deepcopy(entry)
        filtered["predictions"] = valid_predictions
        return filtered


def filter2(fol_predictions, ground_truth, threshold: float = 0.5):
    if len(fol_predictions) == 0:
        return None
    count = {"True": 0, "False": 0, "Uncertain": 0}
    for prediction in fol_predictions:
        label = prediction.get("label")
        if label in count:
            count[label] += 1
    if ground_truth in count and count[ground_truth] / len(fol_predictions) > threshold:
        positive_sample = [prediction for prediction in fol_predictions if prediction.get("label") == ground_truth]
        negative_sample = [prediction for prediction in fol_predictions if prediction.get("label") != ground_truth]
        return positive_sample, negative_sample
    return None


## 6. Baseline 1

In [7]:
def build_label_prompt(example):
    """Prompt used for Basic 1: natural language -> label, no fine-tuning."""
    normalized = normalize_example(example)
    return f"""Given the premises and conclusion, choose exactly one label from: True, False, Uncertain.

Premises:
{normalized['premises']}

Conclusion:
{normalized['conclusion']}

Label:"""


def normalize_label(text):
    raw = str(text).strip()
    cleaned = re.sub(r"[`\"']", " ", raw)
    cleaned = re.sub(r"[^A-Za-z]+", " ", cleaned).strip().lower()
    mapping = {
        "true": "True",
        "false": "False",
        "uncertain": "Uncertain",
        "unknown": "Uncertain",
        "not enough information": "Uncertain",
    }
    if cleaned in mapping:
        return mapping[cleaned]
    tokens = cleaned.split()
    if tokens:
        first = tokens[0]
        if first in mapping:
            return mapping[first]
        if first == "t":
            return "True"
        if first == "f":
            return "False"
        if first == "u":
            return "Uncertain"
    return "Invalid"


LABEL_CANDIDATES = ["True", "False", "Uncertain"]


def score_label_candidate(model, tokenizer, prompt, candidate_label, device, max_source_length=768):
    """Return seq2seq loss for one candidate label. Lower is better."""
    source = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_source_length,
    )
    try:
        target = tokenizer(
            text_target=candidate_label,
            return_tensors="pt",
            truncation=True,
            max_length=8,
        )
    except TypeError:
        with tokenizer.as_target_tokenizer():
            target = tokenizer(
                candidate_label,
                return_tensors="pt",
                truncation=True,
                max_length=8,
            )

    source = {k: v.to(device) for k, v in source.items()}
    labels = target["input_ids"].to(device)

    pad_id = tokenizer.pad_token_id
    if pad_id is not None:
        labels = labels.masked_fill(labels == pad_id, -100)

    with torch.no_grad():
        outputs = model(**source, labels=labels)

    return float(outputs.loss.detach().cpu().item())


def predict_label_by_scoring(model, tokenizer, prompt, device):
    """Basic 1 inference without fine-tuning: score True/False/Uncertain and choose lowest loss."""
    losses = {
        label: score_label_candidate(model, tokenizer, prompt, label, device)
        for label in LABEL_CANDIDATES
    }
    prediction = min(losses, key=losses.get)
    return prediction, losses


def format_accuracy(correct, total):
    if total == 0:
        return "0.0000"
    return f"{correct / total:.4f}"


def save_basic1_group_result(rows, output_path):
    # Official group format: ONLY accuracy + data; each row ONLY id/predict/label.
    group_rows = [{"id": row["id"], "predict": row["predict"], "label": row["label"]} for row in rows]
    correct = sum(1 for row in group_rows if row["predict"] == row["label"])
    payload = {"accuracy": format_accuracy(correct, len(group_rows)), "data": group_rows}
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    return payload


def majority_vote(labels):
    valid = [x for x in labels if x in {"True", "False", "Uncertain"}]
    if not valid:
        return "Invalid"
    return Counter(valid).most_common(1)[0][0]


def save_basic2_group_result(rows, output_path):
    # Official group format: ONLY accuracy + data.
    group_rows = []
    correct = 0
    for row in rows:
        predictions = []
        for pred in row.get("predictions", []):
            label = pred.get("label")
            if label is None:
                label = "Invalid"
            predictions.append({"fol": pred.get("fol", ""), "label": label})
        final_prediction = majority_vote([pred["label"] for pred in predictions])
        correct += int(final_prediction == row["label"])
        group_rows.append({"id": row["id"], "natural": row["natural"], "predictions": predictions, "label": row["label"]})
    payload = {"accuracy": format_accuracy(correct, len(group_rows)), "data": group_rows}
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    return payload


def run_mt5_baseline1(
    model_name="google/mt5-small",
    test_path=MANUAL_TEST_PATH,
    output_path=OUTPUT_DIR / "result_basic_1_mt5.json",
    debug_output_path=OUTPUT_DIR / "debug_basic_1_mt5.json",
    max_samples=None,
    max_new_tokens=8,
    local_files_only=False,
    invalid_fallback=None,
    method="score",
):
    """Run Basic 1 in the same official output format as result_basic_1.json.

    method="score" is the default because raw google/mt5-small free generation often outputs
    sentinel tokens like <extra_id_0>, which makes the result look strange and Invalid.
    Scoring is still direct inference and uses no fine-tuning: it simply compares the loss for
    candidate targets True/False/Uncertain and chooses the lowest-loss label.
    """
    if invalid_fallback is not None and invalid_fallback not in {"True", "False", "Uncertain"}:
        raise ValueError("invalid_fallback must be one of True, False, Uncertain, or None.")
    if method not in {"score", "generate"}:
        raise ValueError('method must be either "score" or "generate".')

    tokenizer = AutoTokenizer.from_pretrained(model_name, local_files_only=local_files_only)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name, local_files_only=local_files_only)

    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token
    if model.config.pad_token_id is None:
        model.config.pad_token_id = tokenizer.pad_token_id
    if model.config.decoder_start_token_id is None:
        model.config.decoder_start_token_id = model.config.pad_token_id

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    rows = load_json(test_path)
    if max_samples is not None:
        rows = rows[:max_samples]

    group_rows = []
    debug_rows = []
    invalid = 0

    for i, row in enumerate(rows):
        normalized = normalize_example(row, i)
        prompt = build_label_prompt(row)

        if method == "score":
            official_prediction, candidate_losses = predict_label_by_scoring(model, tokenizer, prompt, device)
            normalized_prediction = official_prediction
            raw_prediction = official_prediction
        else:
            inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=768)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            with torch.no_grad():
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    num_beams=1,
                    pad_token_id=tokenizer.pad_token_id,
                )
            raw_prediction = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
            normalized_prediction = normalize_label(raw_prediction)
            candidate_losses = None

        invalid += int(normalized_prediction == "Invalid")
        official_prediction = invalid_fallback if normalized_prediction == "Invalid" and invalid_fallback is not None else normalized_prediction

        # Official row must match result_basic_1.json exactly: id, predict, label.
        group_rows.append({
            "id": normalized["id"],
            "predict": official_prediction,
            "label": normalized["label"],
        })

        debug_row = {
            "id": normalized["id"],
            "label": normalized["label"],
            "predict": official_prediction,
            "normalized_prediction": normalized_prediction,
            "raw_prediction": raw_prediction,
            "premises": normalized["premises"],
            "conclusion": normalized["conclusion"],
            "method": method,
        }
        if candidate_losses is not None:
            debug_row["candidate_losses"] = candidate_losses
        debug_rows.append(debug_row)

    payload = save_basic1_group_result(group_rows, output_path)

    if debug_output_path is not None:
        debug_payload = {
            "baseline": "baseline1",
            "model_name": model_name,
            "accuracy": payload["accuracy"],
            "num_examples": len(group_rows),
            "num_invalid": invalid,
            "invalid_fallback": invalid_fallback,
            "official_output_path": str(output_path),
            "method": method,
            "data": debug_rows,
        }
        debug_output_path = Path(debug_output_path)
        debug_output_path.parent.mkdir(parents=True, exist_ok=True)
        debug_output_path.write_text(json.dumps(debug_payload, ensure_ascii=False, indent=2), encoding="utf-8")
        print("Saved debug:", debug_output_path)

    print("Saved official group result:", output_path)
    print("Accuracy:", payload["accuracy"])
    return payloads

### Baseline 1 Tiny Run

In [10]:
baseline1_result = run_mt5_baseline1(
    output_path=OUTPUT_DIR / "result_basic_1_mt5.json",
    debug_output_path=OUTPUT_DIR / "debug_basic_1_mt5.json",
    # Set max_samples=10 for a quick smoke test. Use None for the full manual_test set.
    max_samples=None,
    method="score",
)
baseline1_result

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Saved debug: /kaggle/working/mt5_outputs/debug_basic_1_mt5.json
Saved official group result: /kaggle/working/mt5_outputs/result_basic_1_mt5.json
Accuracy: 0.3806


{'accuracy': '0.3806',
 'data': [{'id': 'manual_0', 'predict': 'Uncertain', 'label': 'Uncertain'},
  {'id': 'manual_1', 'predict': 'Uncertain', 'label': 'Uncertain'},
  {'id': 'manual_2', 'predict': 'False', 'label': 'Uncertain'},
  {'id': 'manual_3', 'predict': 'Uncertain', 'label': 'Uncertain'},
  {'id': 'manual_4', 'predict': 'Uncertain', 'label': 'True'},
  {'id': 'manual_5', 'predict': 'Uncertain', 'label': 'Uncertain'},
  {'id': 'manual_6', 'predict': 'False', 'label': 'Uncertain'},
  {'id': 'manual_7', 'predict': 'Uncertain', 'label': 'Uncertain'},
  {'id': 'manual_8', 'predict': 'False', 'label': 'Uncertain'},
  {'id': 'manual_9', 'predict': 'Uncertain', 'label': 'Uncertain'},
  {'id': 'manual_10', 'predict': 'False', 'label': 'Uncertain'},
  {'id': 'manual_11', 'predict': 'False', 'label': 'Uncertain'},
  {'id': 'manual_12', 'predict': 'False', 'label': 'Uncertain'},
  {'id': 'manual_13', 'predict': 'False', 'label': 'Uncertain'},
  {'id': 'manual_14', 'predict': 'Uncertain', 